# Qwen 27B Uncensored — Free Kaggle + Cloudflare Tunnel
### Free P100 GPU (faster than T4) · No credit card · No port forwarding

**Step 1:** Right panel > Accelerator > **GPU P100** > Save
**Step 2:** Edit > Run All
**Step 3:** Copy `SELF_HOSTED_BASE_URL=...` from the last cell into your JARVIS `.env`

**Keep this session open** while using JARVIS.

In [ ]:
!nvidia-smi

!apt-get update -qq && apt-get install -y -qq git-lfs wget curl build-essential cmake git > /dev/null 2>&1
!pip install -q huggingface-hub
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import subprocess
r = subprocess.run(['/usr/local/bin/cloudflared', '--version'], capture_output=True, text=True)
print(f'cloudflared: {r.stdout.strip()}')

result = subprocess.run(['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'], capture_output=True, text=True)
vram_mb = int(result.stdout.strip().split('.')[0])
vram_gb = vram_mb / 1024
print(f'VRAM: {vram_gb:.1f} GB')
if vram_gb < 14:
    raise RuntimeError(f'Need 14GB+ VRAM but got {vram_gb:.1f}GB. Enable GPU P100!')
print('Cell 1 DONE')

In [ ]:
import os

GGUF_DIR = '/kaggle/working/model'
GGUF_FILE = os.path.join(GGUF_DIR, 'Qwen3.8-27B-Uncensored-noMTP-IQ2_M.gguf')
HF_REPO = 'JonathanColetti/Qwen3.8-27B-Uncensored-GGUF'
HF_FILE = 'Qwen3.8-27B-Uncensored-noMTP-IQ2_M.gguf'

if os.path.exists(GGUF_FILE) and os.path.getsize(GGUF_FILE) > 1e9:
    print(f'GGUF already exists: {os.path.getsize(GGUF_FILE)/1e9:.2f} GB')
else:
    os.makedirs(GGUF_DIR, exist_ok=True)
    print(f'Downloading {HF_FILE} ...')
    print('~10 GB, may take 5-10 min on Kaggle.')
    from huggingface_hub import hf_hub_download
    path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE, local_dir=GGUF_DIR, local_dir_use_symlinks=False)
    actual = os.path.getsize(path)
    print(f'Downloaded: {actual/1e9:.2f} GB')
    if actual < 5e9:
        raise RuntimeError(f'Download too small ({actual/1e9:.2f} GB)')

print(f'Model ready: {GGUF_FILE}')

In [ ]:
import os

LLAMA_DIR = '/kaggle/working/llama.cpp'
BUILD_DIR = os.path.join(LLAMA_DIR, 'build')
SERVER_BIN = os.path.join(BUILD_DIR, 'bin', 'llama-server')

if os.path.exists(SERVER_BIN):
    print('llama.cpp already built, skipping')
else:
    if not os.path.exists(LLAMA_DIR):
        print('Cloning llama.cpp...')
        !git clone https://github.com/ggml-org/llama.cpp.git $LLAMA_DIR
    # P100 = compute capability 6.0
    import os
    os.environ['CUDA_HOME'] = '/usr/local/cuda'
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'
    print('Building with CUDA for P100 (5-8 min)...')
    !cmake -S $LLAMA_DIR -B $BUILD_DIR -DGGML_CUDA=ON -DLLAMA_CURL=ON -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=60 > /kaggle/working/cmake_log.txt 2>&1
    !cmake --build $BUILD_DIR -j$(nproc) --target llama-server > /kaggle/working/build_log.txt 2>&1
    if not os.path.exists(SERVER_BIN):
        print('BUILD FAILED! Last 30 lines:')
        !tail -30 /kaggle/working/build_log.txt
        raise RuntimeError('Build failed')
    print('Built successfully')

print(f'Server binary: {SERVER_BIN}')

In [ ]:
import subprocess, time, urllib.request, os

PORT = 8080

subprocess.run(['fuser', '-k', f'{PORT}/tcp'], capture_output=True)
subprocess.run(['pkill', '-9', '-f', 'llama-server'], capture_output=True)
time.sleep(2)

if not os.path.exists(GGUF_FILE):
    raise RuntimeError(f'Model not found - re-run Cell 2')

print(f'Starting server: {GGUF_FILE}')
print(f'Model size: {os.path.getsize(GGUF_FILE)/1e9:.2f} GB')

server_proc = subprocess.Popen([
    SERVER_BIN, '-m', GGUF_FILE, '-c', '2048', '-ngl', '99',
    '--host', '0.0.0.0', '--port', str(PORT), '--parallel', '2',
    '-ctk', 'q4_0', '-ctv', 'q4_0', '-t', '2'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

print(f'PID={server_proc.pid} on port {PORT}')
print('Loading model (1-2 min on P100)...')

time.sleep(15)
if server_proc.poll() is not None:
    print('SERVER CRASHED!')
    print(server_proc.stdout.read().decode(errors='replace')[-3000:])
    raise RuntimeError('Server crashed')

print('Waiting for model to load...')
ready = False
for i in range(180):
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/v1/models', timeout=2)
        print(f'Ready in ~{(i+1)*2}s')
        ready = True
        break
    except Exception:
        if server_proc.poll() is not None:
            print('CRASHED while loading!')
            print(server_proc.stdout.read().decode(errors='replace')[-3000:])
            raise RuntimeError('Server crashed')
        time.sleep(2)
        if (i + 1) % 15 == 0:
            print(f'  loading... ({(i+1)*2}s)')

if not ready:
    raise RuntimeError('Not ready after 6 min')

print('Cell 4 DONE')

In [ ]:
import re, threading, urllib.request, json, subprocess, time

tunnel_url = [None]

subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(1)

tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)

def watch_tunnel():
    for raw in tunnel_proc.stdout:
        line = raw.decode(errors='replace')
        m = re.search(r'https://([a-z0-9-]+\.trycloudflare\.com)', line)
        if m:
            tunnel_url[0] = m.group(1)
            print()
            print('=' * 55)
            print('TUNNEL: https://' + tunnel_url[0])
            print('=' * 55)
            break

threading.Thread(target=watch_tunnel, daemon=True).start()
print('Waiting for tunnel URL...')

for i in range(60):
    if tunnel_url[0]:
        break
    time.sleep(1)
    if (i + 1) % 10 == 0:
        print(f'  waiting... ({i+1}s)')

if tunnel_url[0]:
    BASE = 'https://' + tunnel_url[0]
else:
    BASE = f'http://localhost:{PORT}'
    print('WARNING: Tunnel URL not found.')

print()
print('*' * 55)
print('COPY THIS INTO YOUR JARVIS .env FILE:')
print(f'SELF_HOSTED_BASE_URL={BASE}')
print('*' * 55)

print('Testing model...')
try:
    payload = json.dumps({'model': 'qwen', 'messages': [{'role': 'user', 'content': 'Say hello in 5 words.'}], 'max_tokens': 50, 'stream': False}).encode()
    req = urllib.request.Request(BASE + '/v1/chat/completions', data=payload, headers={'Content-Type': 'application/json'})
    resp = urllib.request.urlopen(req, timeout=120)
    data = json.loads(resp.read())
    msg = data['choices'][0]['message']['content']
    print(f'Response: {msg}')
    print('ALL WORKING!')
except Exception as e:
    print(f'Error: {e}')
    print('Wait and re-run this cell.')

## Done!

**Keep this session open.** If it disconnects, re-run **Cells 4 and 5 only**.